In [19]:
import pandas as pd

m1=pd.read_csv("Final_Predictions.csv")
m2=pd.read_csv("Generated_Data.csv")




ml_batch = m1[m1["Formula_pretty"].isin(m2["Formula_pretty"])].copy()
print(ml_batch.shape)
ml_batch.to_csv("OUTPUT DATA/Inorganic_Pred.csv")

(23503, 21)


In [22]:
import pandas as pd
import re

def normalize_formula(formula):
    if pd.isna(formula): return ""
    # Extract element-count pairs and sort alphabetically
    pairs = re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', str(formula).replace(" ", ""))
    d = {}
    for el, count in pairs:
        d[el] = d.get(el, 0) + (float(count) if count else 1.0)
    return "".join([f"{el}{int(c) if c==int(c) else c}" for el, c in sorted(d.items())])

# Load and clean
p2 = pd.read_csv('INPUT DATA/perovskite2.csv')
ip = ml_batch

p2['key'] = p2['formula_pretty'].apply(normalize_formula) + "_" + p2['crystal_system'].str.strip().str.capitalize()
ip['key'] = ip['Long_Form'].apply(normalize_formula) + "_" + ip['Crystal_System'].str.strip().str.capitalize()

# Extract intersection
common_keys = set(p2['key']).intersection(set(ip['key']))
ml_real = p2[p2['key'].isin(common_keys)].drop_duplicates('key').sort_values('key')
ml_pred = ip[ip['key'].isin(common_keys)].drop_duplicates('key').sort_values('key')
print(ml_real.shape,ml_pred.shape)

(83, 11) (83, 22)


In [17]:
m1=pd.read_csv("OUTPUT DATA/Generated_Data.csv")
df = pd.read_csv('OUTPUT DATA/Predictions.csv')

df["TEF"]=m1["TEF"]
df.to_csv("OUTPUT DATA/Predictions.csv",index=False)

In [108]:
import pandas as pd

# 1. LOAD DATA
df = pd.read_csv('OUTPUT DATA/Predictions.csv')
print(f"Loaded {len(df)} materials.")




# TEF Goal: 'MAX' for Shields/Detectors, 'TARGET' for Dosimeters
TEF_GOAL = 'MAX' 
# ==========================================
# 3. APPLY FILTERS (The "Optimization")
# ==========================================

# Step A: Stability Filter (The "Goldschmidt" Screen)
# We only keep materials that are likely to form stable perovskites
stable_df = df[
    (df['Formation_Energy'] < 0) &
    (df["Is_abundant"]==True)
].copy()

print(f" -> {len(stable_df)} materials passed Stability screening.")

# Step B: Band Gap Filter (The "Semiconductor" Screen)

#BANDGAP 1

# Application Constraints (e.g., X-ray Detector)
# Band Gap: 0.5 - 2.5 eV is ideal for room-temp detectors (low noise)
MIN_GAP = 0.5
MAX_GAP = 2.0

optimized1 = stable_df[
    stable_df['Band_Gap'].between(MIN_GAP, MAX_GAP) &
    stable_df["Total_cost"].between(0,50)
].copy()

print(f" -> {len(optimized1)} materials passed Band Gap 1 screening.")


#BANDGAP 2
# Application Constraints (e.g., X-ray Detector)
# Band Gap: 2.0 - 4.5 eV is ideal for room-temp detectors (low noise)
MIN_GAP = 2.0
MAX_GAP = 4.5

optimized2 = stable_df[
    stable_df['Band_Gap'].between(MIN_GAP, MAX_GAP) &
    stable_df["Total_cost"].between(0,50)
].copy()

print(f" -> {len(optimized2)} materials passed Band Gap 2 screening.")


#BANDGAP 3
# Application Constraints (e.g., X-ray Detector)
# Band Gap: 5.4 - 7.5 eV is ideal for room-temp detectors (low noise)
MIN_GAP = 5.4
MAX_GAP = 7.5

optimized3 = stable_df[
    stable_df['Band_Gap'].between(MIN_GAP, MAX_GAP) &
    stable_df["Total_cost"].between(0,50)
].copy()

print(f" -> {len(optimized3)} materials passed Band Gap 3 screening.\n")


# ==========================================
# 4. RANKING & SCORING
# ==========================================
print("Optimization Goal: Maximize TEF (High Stopping Power) sorted by first 100")
final1 = optimized1.sort_values(by=['TEF',"Total_cost"], ascending=[False,True]).head(100)
final2 = optimized2.sort_values(by=['TEF',"Total_cost"], ascending=[False,True]).head(100)
final3 = optimized3.sort_values(by=['TEF',"Total_cost"], ascending=[False,True]).head(100)

# ==========================================
# 5. RESULTS
# ==========================================
# Save the "Best" candidates
output_file = "OUTPUT DATA/Optimized1.csv"
final1.to_csv(output_file, index=False)

output_file = "OUTPUT DATA/Optimized2.csv"
final2.to_csv(output_file, index=False)

output_file = "OUTPUT DATA/Optimized3.csv"
final3.to_csv(output_file, index=False)

print("\nTOP 5 OPTIMIZED CANDIDATES:")
cols = ['Formula_pretty','TEF', 'Band_Gap','Total_cost']
print(final1[cols].head(5))

print(f"\nTop 100 candidates saved to optimized files")

Loaded 21769 materials.
 -> 5140 materials passed Stability screening.
 -> 492 materials passed Band Gap 1 screening.
 -> 530 materials passed Band Gap 2 screening.
 -> 6 materials passed Band Gap 3 screening.

Optimization Goal: Maximize TEF (High Stopping Power) sorted by first 100

TOP 5 OPTIMIZED CANDIDATES:
      Formula_pretty        TEF  Band_Gap  Total_cost
17671     Ba2MnWO3O3  35.813530  1.796341      38.074
17672     Ba2MnWO3O3  35.813530  1.796341      38.074
17673     Ba2MnWO3O3  35.813530  1.796341      38.074
17478     Ba2CrWO3O3  35.681087  1.793020      45.654
17479     Ba2CrWO3O3  35.681087  1.793020      45.654

Top 100 candidates saved to optimized files


In [98]:
print(max(final1["TEF"]),min(final1["TEF"]))
print(max(final2["TEF"]),min(final2["TEF"]))
print(max(final3["TEF"]),min(final3["TEF"]))

35.81352967 22.78182751
37.55657881 22.55603164
21.09103555 15.67206803


In [56]:
ELEMENT_COSTS = {
    'Cs': 61800.0, 'Rb': 15500.0, 'K': 13.6, 'Na': 2.57, 'Li': 85.6, "Re":87300,
    'Ca': 2.21, 'Sr': 6.68, 'Ba': 0.246,"Mo":40.1,"Mn":1.82,"V":385,"Ru":10600,"Y":31.0,
    'Rh': 147000.0, 'Pd': 49500.0, 'Pt': 27800.0, 'Au': 75430.0, 'Ag': 521.0,"Ta":312,"Zr":37.1,
    'Sc': 3460.0, 'Ge': 1010.0, 'In': 167.0, 'Ga': 148.0, 'Sn': 18.7,"Bi":6.36,"Mg":2.32,
    'Cu': 6.0, 'Ni': 13.9, 'Zn': 2.55, 'Ti': 11.5, 'Co': 32.8, 'Fe': 0.424,"Nb":81.6,"W":35.3,"Tc":100000.0,
    'I': 35.0, 'Br': 4.390, 'Cl': 0.082, 'F': 2.16, 'O': 0.154, 'S': 0.0926, 'Se': 21.4,"Hf":900,"Al":1.79,"Cr":9.4
}

#As found in wikipedia USD/kg

In [104]:
import regex as re
def total_cost(row):
    composition=dict(re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', row["Formula_pretty"]))
    for i in composition:
        if composition[i]=="":
            composition[i]=1
        else:
            composition[i]=float(composition[i])

    total=0.0
    for i in composition:
        total+=ELEMENT_COSTS[i]*composition[i]
    return total

abundance=pd.read_csv("INPUT DATA/Abundance.csv")
abundant_el=dict()

def min_abundance(row):

    composition=dict(re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', row["Formula_pretty"]))
    for i in composition:
        if composition[i]=="":
            composition[i]=1
        else:
            composition[i]=float(composition[i])
            
    all_elements=set(row["A_ions"].split("; ")+row["B_ions"].split("; ")+row["C_ions"].split("; "))

    minv=float("inf")

    for i in composition:
        if i in list(abundance["Element"]):
            r=abundance[abundance["Element"]==i]
            v=r["Abundance_PPM"].iloc[0]
            if v=="NA": continue
            print(i,"divided by ",composition[i])
            v=float(v)/composition[i]
            if i not in abundant_el:
                abundant_el[i]=v
            if v<2.30: #min abuncance in ppm
                return False
    
    return True

In [105]:
df["Total_cost"]=df.apply(total_cost,axis=1)
df["Is_abundant"]=df.apply(min_abundance,axis=1)

df.to_csv("OUTPUT DATA/Predictions.csv",index=False)

Rb divided by  1
Ge divided by  1
Rb divided by  1
Pd divided by  1
Rb divided by  1
Ti divided by  1
Br divided by  3.0
Rb divided by  1
Sr divided by  1
I divided by  3.0
Rb divided by  1
Sr divided by  1
Cl divided by  3.0
Rb divided by  1
Ni divided by  1
F divided by  3.0
Rb divided by  1
Bi divided by  1
Rb divided by  1
Zn divided by  1
F divided by  3.0
Rb divided by  1
Mg divided by  1
F divided by  3.0
Rb divided by  1
Ca divided by  1
I divided by  3.0
Rb divided by  1
Ca divided by  1
Cl divided by  3.0
Rb divided by  1
Ca divided by  1
Br divided by  3.0
Rb divided by  1
Ag divided by  1
Rb divided by  1
Ag divided by  1
Rb divided by  1
Ag divided by  1
Rb divided by  1
Nb divided by  1
O divided by  3.0
Rb divided by  1
W divided by  1
O divided by  3.0
Rb divided by  1
Tc divided by  1
O divided by  3.0
Rb divided by  1
Co divided by  1
F divided by  3.0
Rb divided by  1
Ta divided by  1
Rb divided by  1
Cr divided by  1
F divided by  3.0
Rb divided by  1
Cu divided by 

In [107]:
print(df["Is_abundant"].value_counts())

Is_abundant
False    16629
True      5140
Name: count, dtype: int64


In [96]:
print(max(df["Total_cost"]),min(df["Total_cost"]))

print(max(abundant_el.values()),min(abundant_el.values()))

320113.632 1.132
474000.0 0.0002


In [50]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

elements = ["H", "He", "Li", "Be", "B", "C", "N", "O", "F", "Ne", "Na", "Mg", "Al", "Si", "P", "S", "Cl", "Ar", "K", "Ca", "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Ga", "Ge", "As", "Se", "Br", "Kr", "Rb", "Sr", "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd", "In", "Sn", "Sb", "Te", "I", "Xe", "Cs", "Ba", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg", "Tl", "Pb", "Bi", "Po", "At", "Rn", "Fr", "Ra", "Ac", "Th", "Pa", "U", "Np", "Pu", "Am", "Cm", "Bk", "Cf", "Es", "Fm", "Md", "No", "Lr", "Rf", "Db", "Sg", "Bh", "Hs", "Mt", "Ds", "Rg", "Cn", "Nh", "Fl", "Mc", "Lv", "Ts", "Og"]
base_url = "https://environmentalchemistry.com/yogi/periodic/"
results = []

for symbol in elements:
    url = f"{base_url}{symbol}.html"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        abundance_header = soup.find('b', string=re.compile(r"Abundance of", re.I))
        
        if abundance_header:
            crust_label = abundance_header.find_next('b', string=re.compile(r"Earth's Crust", re.I))
            
            if crust_label:
                # FIXED: Grab ONLY the text immediately following the <b> tag
                # This ignores the rest of the broken list (Seawater, Sun, etc.)
                sibling_text = str(crust_label.next_sibling)
                
                # sibling_text will look like: "/p.p.m.: 5600"
                value = sibling_text.split(':')[-1].strip().replace(",", "")
                if value=="N/A":
                    results.append({"Element": symbol, "Abundance_PPM": "NA"})
                else:
                    results.append({"Element": symbol, "Abundance_PPM": float(value)})
                print(f"Processed {symbol}: {value}")
            else:
                results.append({"Element": symbol, "Abundance_PPM": "NA"})
                print(f"Processed {symbol}: No crust")
        else:
            results.append({"Element": symbol, "Abundance_PPM": "NA"})
            print(f"Processed {symbol}: No abundance")
            
    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")
        
    time.sleep(1)

df = pd.DataFrame(results)
print("\nFinal Results:")
print(df)
df.to_csv("INPUT DATA/Abundance.csv")

Processed H: N/A
Processed He: 0.008
Processed Li: 20
Processed Be: 2.6
Processed B: 950
Processed C: 480
Processed N: 25
Processed O: 474000
Processed F: 950
Processed Ne: 0.00007
Processed Na: 23000
Processed Mg: 23000
Processed Al: 82000
Processed Si: 277100
Processed P: 1000
Processed S: 260
Processed Cl: 130
Processed Ar: 1.2
Processed K: 21000
Processed Ca: 41000
Processed Sc: 16
Processed Ti: 5600
Processed V: 160
Processed Cr: 100
Processed Mn: 950
Processed Fe: 41000
Processed Co: 20
Processed Ni: 80
Processed Cu: 50
Processed Zn: 75
Processed Ga: 18
Processed Ge: 1.8
Processed As: 1.5
Processed Se: 0.05
Processed Br: 0.37
Processed Kr: 0.00001
Processed Rb: 90
Processed Sr: 370
Processed Y: 30
Processed Zr: 190
Processed Nb: 20
Processed Mo: 1.5
Processed Tc: N/A
Processed Ru: 0.001
Processed Rh: 0.0002
Processed Pd: 0.0006
Processed Ag: 0.07
Processed Cd: 0.11
Processed In: 0.049
Processed Sn: 2.2
Processed Sb: 0.2
Processed Te: 0.005
Processed I: 0.14
Processed Xe: 0.000002

In [61]:
print(min(abundance["Abundance_PPM"]))

nan
